# Setup

**Conjunction–CME Catalog Matching**

Assemble continuous multi-partner conjunctions from the supplied daily table,
then find catalog CMEs within each conjunction interval with ±24-hour tolerance.
Run with the `icme3.12-metal` kernel. All inputs and new results live under
`ROOT_FOLDER` in the attached Mercury storm Google Drive folder.

| Term | Meaning |
| --- | --- |
| Conjunction Day | One daily geometry sample from the input table. |
| Conjunction | A continuous run of consecutive input dates, even when partners change. |
| Partner | A planet or spacecraft named in the input table's final column. |
| CME | A catalog entry at MESSENGER or a partner, including shock-only CMEs. |
| Match | One conjunction linked to one catalog CME within the tolerance. |

MESSENGER CMEs and partner CMEs are matched independently to the conjunction.
No MESSENGER-CME/partner-CME association is required or inferred.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def show_table(table):
    """Display UTC timestamps without offsets while keeping timezone-aware calculations."""
    preview = table.copy()
    for column in preview.select_dtypes(include=["datetime", "datetimetz"]):
        preview[column] = preview[column].dt.strftime("%Y-%m-%d %H:%M:%S")
    display(preview.astype(object).where(preview.notna(), ""))

In [2]:
ROOT_FOLDER = Path("/Users/aosh/Library/CloudStorage/GoogleDrive-anton.o.shiryaev@gmail.com/.shortcut-targets-by-id/1eEjUfN4R9n8Qq1Fd8cc7SUKpQcdQkb3g/Mercury storm")
INPUT_FOLDER = ROOT_FOLDER / "Inputs"
RESULTS_FOLDER = ROOT_FOLDER / "Conjunction Analysis"
TOLERANCE = pd.Timedelta(hours=24)

# Assemble Conjunctions

A conjunction ID belongs to one uninterrupted run of input dates. Partners may
overlap, appear, or disappear without changing that ID. For example, the previous
CONJ-003 and CONJ-004 now form one CONJ-003, from 2011-05-17 to 2011-06-08.

Keep each partner's actual participation intervals inside the conjunction.
Partner CMEs are matched to those intervals; MESSENGER CMEs are matched to the
whole conjunction. Starts and ends are the first and last supplied midnight
timestamps, with no extra day added. Times are interpreted as UTC.

The supplied CSV and workbook filenames are preserved. The workbook's older
Earth-CME selection does not control this analysis.

In [3]:
daily_table = pd.read_csv(INPUT_FOLDER / "messenger_conjunctions.csv")
partner_column = daily_table.columns[-1]
daily_table["Date"] = pd.to_datetime(daily_table["Date"], utc=True)
daily_table = daily_table.sort_values("Date").reset_index(drop=True)

# The ID follows continuous input dates, regardless of partner changes.
period = daily_table["Date"].diff().ne(pd.Timedelta(days=1)).cumsum()
daily_table["conjunction_id"] = [f"CONJ-{number:03d}" for number in period]
conjunctions = daily_table.groupby("conjunction_id", as_index=False).agg(
    conjunction_start=("Date", "min"), conjunction_end=("Date", "max"), days=("Date", "size")
)

conjunction_days = daily_table.assign(partner=daily_table[partner_column].str.split(";")).explode("partner")
partner_names = {"Earth": "Earth", "Venus": "Venus", "STEREO_A": "STEREO-A", "STEREO_B": "STEREO-B"}
conjunction_days["partner"] = [partner_names[value.strip()] for value in conjunction_days["partner"]]
partners = conjunction_days.groupby("conjunction_id")["partner"].agg(lambda values: "; ".join(sorted(set(values))))
conjunctions = conjunctions.merge(partners.rename("partners"), on="conjunction_id")

# A partner may participate in only part of a multi-partner conjunction.
conjunction_days = conjunction_days.sort_values(["conjunction_id", "partner", "Date"]).reset_index(drop=True)
keys = ["conjunction_id", "partner"]
new_interval = conjunction_days.groupby(keys)["Date"].diff().ne(pd.Timedelta(days=1))
conjunction_days["run"] = new_interval.groupby([conjunction_days[key] for key in keys]).cumsum()
partner_intervals = conjunction_days.groupby(keys + ["run"], as_index=False).agg(
    matching_start=("Date", "min"), matching_end=("Date", "max")
).drop(columns="run")
conjunction_days = conjunction_days.drop(columns="run")
show_table(conjunctions)

,conjunction_id,conjunction_start,conjunction_end,days,partners
0,CONJ-001,2011-03-18 00:00:00,2011-03-20 00:00:00,3,STEREO-B
1,CONJ-002,2011-04-03 00:00:00,2011-04-19 00:00:00,17,Earth
2,CONJ-003,2011-05-17 00:00:00,2011-06-08 00:00:00,23,STEREO-A; Venus
3,CONJ-004,2011-06-28 00:00:00,2011-07-12 00:00:00,15,STEREO-B
4,CONJ-005,2011-08-09 00:00:00,2011-08-23 00:00:00,15,Earth
5,CONJ-006,2011-09-07 00:00:00,2011-09-14 00:00:00,8,STEREO-A
6,CONJ-007,2011-10-03 00:00:00,2011-11-14 00:00:00,43,STEREO-B; Venus
7,CONJ-008,2011-12-01 00:00:00,2011-12-08 00:00:00,8,Earth
8,CONJ-009,2011-12-25 00:00:00,2012-01-11 00:00:00,18,STEREO-A
9,CONJ-010,2012-02-21 00:00:00,2012-02-28 00:00:00,8,STEREO-B


# Read Catalog CMEs

Our **CME start** is HELIO4CAST `mo_start_time` or SINP `T_start`.
HELIO4CAST `icme_start_time` is its shock/disturbance arrival field; SINP uses
`T_shock`. All times are interpreted as UTC. The solar eruption time `T_source`
is not used as an in-situ arrival.

A SINP CME with missing `T_start` uses `T_shock` for matching and is labeled
**Shock-Only CME**. Its CME-body start and end remain missing. `match_field`
and `cme_time` show which boundary is actually used.

The complete HELIO4CAST v2.3 source snapshot avoids year-cutoff ambiguity between
shock and MO starts. Its original filename and parameter header are preserved
under `Inputs/HELIO4CAST`. Only the five requested spacecraft are selected.
The per-spacecraft 2011–2015 copies are available there as well.

In [4]:
helio = pd.read_csv(INPUT_FOLDER / "HELIO4CAST" / "HELIO4CAST_ICMECAT_v23.csv")
locations = {"MESSENGER": "Mercury", "Wind": "Earth", "VEX": "Venus", "STEREO-A": "STEREO-A", "STEREO-B": "STEREO-B"}
helio = helio[helio["sc_insitu"].isin(locations)].copy()
for column in ("icme_start_time", "mo_start_time", "mo_end_time"):
    helio[column] = pd.to_datetime(helio[column], utc=True)

helio_cmes = helio[["icmecat_id", "sc_insitu", "mo_start_time", "mo_end_time", "icme_start_time", "mo_bmax", "mo_speed_mean"]].rename(columns={
    "icmecat_id": "cme_id", "sc_insitu": "observer", "mo_start_time": "cme_start",
    "mo_end_time": "cme_end", "icme_start_time": "shock_time", "mo_bmax": "bmax", "mo_speed_mean": "vmean",
})
helio_cmes["catalog"] = "HELIO4CAST"
helio_cmes["location"] = helio_cmes["observer"].map(locations)
helio_cmes["cme_type"] = "CME"
helio_cmes["match_field"] = "mo_start_time"
helio_cmes["cme_time"] = helio_cmes["cme_start"]
# Mean speed covers the same magnetic-obstacle interval as the body boundaries.
helio_cmes["dst"] = float("nan")

In [5]:
sinp = pd.read_csv(INPUT_FOLDER / "merged_icme_short.csv")
for column in ("T_start", "T_end", "T_shock"):
    sinp[column] = pd.to_datetime(sinp[column], utc=True)

sinp_cmes = sinp[["T_start", "T_end", "T_shock", "cat", "v_average_omni_body", "dst_min_omni_body"]].rename(columns={
    "T_start": "cme_start", "T_end": "cme_end", "T_shock": "shock_time",
    "v_average_omni_body": "vmean", "dst_min_omni_body": "dst",
})
# Component means cannot supply a maximum of the magnetic-field magnitude.
sinp_cmes["bmax"] = float("nan")
sinp_cmes["cme_id"] = "SINP-" + sinp["Unnamed: 0"].astype(str)
sinp_cmes["observer"] = "Earth (SINP)"
sinp_cmes["catalog"] = "SINP"
sinp_cmes["location"] = "Earth"
sinp_cmes["cme_type"] = "CME"
sinp_cmes["match_field"] = "T_start"
sinp_cmes["cme_time"] = sinp_cmes["cme_start"]

# Shock-only CMEs have their own match boundary; do not invent a body start.
shock_only = sinp_cmes["cme_start"].isna()
sinp_cmes.loc[shock_only, "cme_type"] = "Shock-Only CME"
sinp_cmes.loc[shock_only, "match_field"] = "T_shock"
sinp_cmes.loc[shock_only, "cme_time"] = sinp_cmes.loc[shock_only, "shock_time"]

cmes = pd.concat([helio_cmes, sinp_cmes], ignore_index=True)
# Keep only catalog CMEs that could match the supplied conjunction period.
cmes = cmes[cmes["cme_time"].between(
    conjunctions["conjunction_start"].min() - TOLERANCE,
    conjunctions["conjunction_end"].max() + TOLERANCE,
)].copy()
show_table(cmes.groupby(["observer", "catalog", "cme_type", "match_field"]).size().rename("cmes").reset_index())

,observer,catalog,cme_type,match_field,cmes
0,Earth (SINP),SINP,CME,T_start,199
1,Earth (SINP),SINP,Shock-Only CME,T_shock,22
2,MESSENGER,HELIO4CAST,CME,mo_start_time,64
3,STEREO-A,HELIO4CAST,CME,mo_start_time,131
4,STEREO-B,HELIO4CAST,CME,mo_start_time,103
5,VEX,HELIO4CAST,CME,mo_start_time,58
6,Wind,HELIO4CAST,CME,mo_start_time,88


# Match Conjunctions to CMEs

Use the inclusive ±24-hour tolerance around each matching interval. MESSENGER
uses the whole conjunction; each partner uses its participation interval(s).
Keep one match per conjunction, catalog, and CME ID. A CME supported by several
participation intervals is counted once, retaining the nearest interval.

`offset_hours` is zero inside the matching interval, negative before its start,
and positive after its end. MESSENGER CMEs and partner CMEs remain independent
conjunction matches.

In [6]:
# Build MESSENGER and partner windows under the shared conjunction IDs.
messenger_windows = conjunctions.assign(
    location="Mercury", matching_start=conjunctions["conjunction_start"],
    matching_end=conjunctions["conjunction_end"],
)
partner_windows = partner_intervals.rename(columns={"partner": "location"}).merge(
    conjunctions, on="conjunction_id"
)
windows = pd.concat([messenger_windows, partner_windows], ignore_index=True)
matches = windows.merge(cmes, on="location")
in_window = matches["cme_time"].between(
    matches["matching_start"] - TOLERANCE, matches["matching_end"] + TOLERANCE
)
matches = matches[in_window].copy()
matches["role"] = "Partner"
matches.loc[matches["observer"].eq("MESSENGER"), "role"] = "MESSENGER"

# Signed distance to the relevant window, with zero for CMEs inside it.
matches["offset_hours"] = 0.0
before = matches["cme_time"] < matches["matching_start"]
after = matches["cme_time"] > matches["matching_end"]
matches.loc[before, "offset_hours"] = (
    matches.loc[before, "cme_time"] - matches.loc[before, "matching_start"]
).dt.total_seconds() / 3600
matches.loc[after, "offset_hours"] = (
    matches.loc[after, "cme_time"] - matches.loc[after, "matching_end"]
).dt.total_seconds() / 3600

# Several windows for the same partner must not duplicate a conjunction–CME match.
matches = matches.loc[matches["offset_hours"].abs().sort_values(kind="stable").index]
matches = matches.drop_duplicates(["conjunction_id", "catalog", "cme_id"])
matches = matches.sort_values(["conjunction_start", "role", "observer", "cme_time"]).reset_index(drop=True)
show_table(matches[[
    "conjunction_id", "partners", "conjunction_start", "conjunction_end", "role", "observer", "catalog",
    "cme_id", "cme_type", "match_field", "cme_time", "cme_start", "cme_end", "offset_hours",
]])

,conjunction_id,partners,conjunction_start,conjunction_end,role,observer,catalog,cme_id,cme_type,match_field,cme_time,cme_start,cme_end,offset_hours
0,CONJ-002,Earth,2011-04-03 00:00:00,2011-04-19 00:00:00,Partner,Earth (SINP),SINP,SINP-838,CME,T_start,2011-04-09 18:00:00,2011-04-09 18:00:00,2011-04-10 02:00:00,0.0
1,CONJ-003,STEREO-A; Venus,2011-05-17 00:00:00,2011-06-08 00:00:00,MESSENGER,MESSENGER,HELIO4CAST,ICME_MES_WINSLOW_20110519_01,CME,mo_start_time,2011-05-19 15:00:00,2011-05-19 15:00:00,2011-05-20 10:36:00,0.0
2,CONJ-003,STEREO-A; Venus,2011-05-17 00:00:00,2011-06-08 00:00:00,MESSENGER,MESSENGER,HELIO4CAST,ICME_MES_WINSLOW_20110604_01,CME,mo_start_time,2011-06-04 16:15:00,2011-06-04 16:15:00,2011-06-05 00:43:00,0.0
3,CONJ-003,STEREO-A; Venus,2011-05-17 00:00:00,2011-06-08 00:00:00,MESSENGER,MESSENGER,HELIO4CAST,ICME_MES_WINSLOW_20110605_01,CME,mo_start_time,2011-06-05 04:29:00,2011-06-05 04:29:00,2011-06-05 12:45:00,0.0
4,CONJ-003,STEREO-A; Venus,2011-05-17 00:00:00,2011-06-08 00:00:00,MESSENGER,MESSENGER,HELIO4CAST,ICME_MES_WINSLOW_20110608_01,CME,mo_start_time,2011-06-08 05:39:00,2011-06-08 05:39:00,2011-06-08 10:06:00,5.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,CONJ-034,STEREO-A; STEREO-B; Venus,2014-07-24 00:00:00,2014-08-19 00:00:00,Partner,STEREO-B,HELIO4CAST,ICME_STEREO_B_JIAN_20140811_01,CME,mo_start_time,2014-08-11 06:20:00,2014-08-11 06:20:00,2014-08-13 14:47:00,0.0
99,CONJ-034,STEREO-A; STEREO-B; Venus,2014-07-24 00:00:00,2014-08-19 00:00:00,Partner,VEX,HELIO4CAST,ICME_VEX_MOESTL_20140729_01,CME,mo_start_time,2014-07-29 15:31:00,2014-07-29 15:31:00,2014-07-30 15:02:00,0.0
100,CONJ-036,STEREO-A; STEREO-B; Venus,2014-11-23 00:00:00,2015-01-05 00:00:00,MESSENGER,MESSENGER,HELIO4CAST,ICME_MES_MOESTL_20141214_01,CME,mo_start_time,2014-12-14 09:52:00,2014-12-14 09:52:00,2014-12-14 21:41:00,0.0
101,CONJ-036,STEREO-A; STEREO-B; Venus,2014-11-23 00:00:00,2015-01-05 00:00:00,MESSENGER,MESSENGER,HELIO4CAST,ICME_MES_MOESTL_20141214_02,CME,mo_start_time,2014-12-14 23:37:00,2014-12-14 23:37:00,2014-12-15 04:37:00,0.0


# Supplement Missing Earth Parameters

`Inputs/OMNI.parquet` contains ICME_list's complete hourly OMNI series. It was
converted from the copied `omni_v_dst.pkl` without resampling or removing rows.
`merged_icme_omni.csv` is the full merged SINP snapshot, also copied here for
completeness. Matching continues to use the existing `merged_icme_short.csv`
snapshot so the selected CMEs and their IDs remain fixed.

Fill only missing **Earth-CME** Bmax and Dst values. Preserve populated
HELIO4CAST/SINP values. OMNI is near-Earth data and cannot fill MESSENGER,
Venus Express, or STEREO measurements.

Use ICME_list's `process_omni` / `GetParFromOMNI` parameter windows: one hour
before the body start through one hour after the body end, inclusive. A
shock-only CME uses one hour before the shock through 21 hours after it
(the producer's 20-hour fallback plus padding). A known body start with no
end uses the producer's start + 20-hour upper boundary.

OMNI Bmax is `max(sqrt(bx² + by² + bz²))` across complete hourly GSE vectors;
Dst is the minimum hourly Dst. Incomplete vectors yield missing magnitude.
No interpolation, gap filling, or CME averages are used to estimate peaks.
This Bmax is a maximum of **hourly vector magnitudes**, distinct from both
the scalar mean of magnetic magnitude and HELIO4CAST's higher-cadence MO peak.
[NASA OMNI parameter definitions](https://omniweb.gsfc.nasa.gov/html/ow_data.html)
distinguish the scalar and vector averages.

**OMNI Supplements** records each window, valid sample counts, derived values,
and which missing parameters were filled. Rows without an Earth CME retain
`Missing` for Dst because they have no CME window.

In [7]:
omni = pd.read_parquet(INPUT_FOLDER / "OMNI.parquet").set_index("date").sort_index()
omni.index = pd.to_datetime(omni.index, utc=True)

# Mask documented OMNI fill values and require all three components for |B|.
field_columns = ["bx", "by", "bz"]
omni[field_columns] = omni[field_columns].mask(omni[field_columns].eq(999.9))
omni["dst"] = omni["dst"].mask(omni["dst"].eq(99999))
omni["b_magnitude"] = omni[field_columns].pow(2).sum(axis=1, min_count=3).pow(0.5)

matches["bmax_source"] = "Missing"
matches.loc[matches["bmax"].notna(), "bmax_source"] = "HELIO4CAST mo_bmax"
matches["dst_source"] = "Missing"
matches.loc[matches["dst"].notna(), "dst_source"] = "SINP dst_min_omni_body"
supplements = []
earth_cmes = matches[matches["location"].eq("Earth") & (matches["bmax"].isna() | matches["dst"].isna())]
for index, cme in earth_cmes.iterrows():
    # Reproduce the existing ICME_list window rules without changing boundaries.
    start = cme["cme_start"] if pd.notna(cme["cme_start"]) else cme["shock_time"]
    window_start = start - pd.Timedelta(hours=1)
    if pd.notna(cme["cme_end"]):
        window_end = cme["cme_end"] + pd.Timedelta(hours=1)
    elif pd.notna(cme["cme_start"]):
        window_end = start + pd.Timedelta(hours=20)
    else:
        window_end = start + pd.Timedelta(hours=21)
    samples = omni.loc[window_start:window_end]
    bmax = samples["b_magnitude"].max()
    dst = samples["dst"].min()
    fill_bmax = pd.isna(cme["bmax"]) and pd.notna(bmax)
    fill_dst = pd.isna(cme["dst"]) and pd.notna(dst)
    if fill_bmax:
        matches.loc[index, "bmax"] = bmax
        matches.loc[index, "bmax_source"] = "OMNI maximum hourly vector magnitude"
    if fill_dst:
        matches.loc[index, "dst"] = dst
        matches.loc[index, "dst_source"] = "OMNI minimum hourly Dst"
    supplements.append({
        "conjunction_id": cme["conjunction_id"], "catalog": cme["catalog"], "cme_id": cme["cme_id"],
        "window_start": window_start, "window_end": window_end,
        "omni_samples": len(samples), "valid_b_samples": samples["b_magnitude"].count(),
        "valid_dst_samples": samples["dst"].count(), "omni_bmax": bmax, "omni_dst_min": dst,
        "previous_bmax": cme["bmax"], "previous_dst": cme["dst"],
        "filled_bmax": fill_bmax, "filled_dst": fill_dst,
    })
omni_supplements = pd.DataFrame(supplements)
show_table(omni_supplements)

/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:239: IOError: sysctlbyname failed for 'hw.l1dcachesize'. Detail: [errno 1] Operation not permitted
/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:239: IOError: sysctlbyname failed for 'hw.l2cachesize'. Detail: [errno 1] Operation not permitted
/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:239: IOError: sysctlbyname failed for 'hw.l3cachesize'. Detail: [errno 1] Operation not permitted
/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:239: IOError: sysctlbyname failed for 'hw.optional.neon'. Detail: [errno 1] Operation not permitted


,conjunction_id,catalog,cme_id,window_start,window_end,omni_samples,valid_b_samples,valid_dst_samples,omni_bmax,omni_dst_min,previous_bmax,previous_dst,filled_bmax,filled_dst
0,CONJ-002,SINP,SINP-838,2011-04-09 17:00:00,2011-04-10 03:00:00,11,11,11,4.633573,-18.0,,-18.0,True,False
1,CONJ-008,SINP,SINP-882,2011-12-02 17:00:00,2011-12-03 08:00:00,16,16,16,7.619055,-29.0,,-29.0,True,False
2,CONJ-008,SINP,SINP-883,2011-12-04 07:00:00,2011-12-04 21:00:00,15,15,15,3.32265,-23.0,,-23.0,True,False
3,CONJ-012,SINP,SINP-902,2012-03-15 16:00:00,2012-03-16 11:00:00,20,20,20,14.707141,-80.0,,-80.0,True,False
4,CONJ-012,SINP,SINP-903,2012-03-22 07:00:00,2012-03-23 01:00:00,19,19,19,9.708759,-19.0,,-19.0,True,False
5,CONJ-012,HELIO4CAST,ICME_Wind_NASA_20120315_01,2012-03-15 15:04:00,2012-03-16 15:10:00,24,24,24,14.707141,-80.0,16.6,,False,True
6,CONJ-015,SINP,SINP-922,2012-07-22 10:00:00,2012-07-23 17:00:00,32,32,32,6.723095,-14.0,,-14.0,True,False
7,CONJ-015,SINP,SINP-923,2012-07-26 11:00:00,2012-07-28 05:00:00,43,43,43,12.707085,5.0,,5.0,True,False
8,CONJ-018,SINP,SINP-938,2012-11-13 05:00:00,2012-11-14 02:00:00,22,22,22,22.731696,-71.0,,-71.0,True,False
9,CONJ-018,SINP,SINP-939,2012-11-13 07:00:00,2012-11-14 04:00:00,22,22,22,22.731696,-94.0,,-94.0,True,False


# Compare Earth Catalogs

For Earth conjunctions, compare SINP and Wind catalog boundaries using ±24 hours:

| SINP CME | SINP Boundary | Wind HELIO4CAST Boundary |
| --- | --- | --- |
| CME with a body start | `T_start` | `mo_start_time` |
| Shock-only CME | `T_shock` | `icme_start_time` |

Both boundaries must also fall inside the same Earth conjunction's tolerated
interval. Wind's shock/disturbance boundary can support a shock comparison even
when its MO start falls outside that interval. This comparison is independent
of MESSENGER and preserves multiple catalog possibilities.

In [8]:
sinp_matches = matches[matches["catalog"].eq("SINP")][[
    "conjunction_id", "conjunction_start", "conjunction_end", "matching_start", "matching_end",
    "cme_id", "cme_type", "match_field", "cme_time"
]].rename(columns={"cme_id": "sinp_cme_id", "match_field": "sinp_field", "cme_time": "sinp_time"})
wind = helio_cmes[helio_cmes["observer"].eq("Wind")][[
    "cme_id", "cme_start", "shock_time"
]].rename(columns={"cme_id": "wind_cme_id", "cme_start": "wind_start", "shock_time": "wind_shock"})
earth_agreement = sinp_matches.merge(wind, how="cross")

# Compare like boundaries: body to body, or shock to shock for shock-only CMEs.
use_shock = earth_agreement["sinp_field"].eq("T_shock")
earth_agreement["wind_field"] = "mo_start_time"
earth_agreement.loc[use_shock, "wind_field"] = "icme_start_time"
earth_agreement["wind_time"] = earth_agreement["wind_start"].where(~use_shock, earth_agreement["wind_shock"])
same_boundary_time = (earth_agreement["sinp_time"] - earth_agreement["wind_time"]).abs() <= TOLERANCE
wind_in_conjunction = earth_agreement["wind_time"].between(
    earth_agreement["matching_start"] - TOLERANCE,
    earth_agreement["matching_end"] + TOLERANCE,
)
earth_agreement = earth_agreement[same_boundary_time & wind_in_conjunction].copy()
earth_agreement["sinp_minus_wind_hours"] = (
    earth_agreement["sinp_time"] - earth_agreement["wind_time"]
).dt.total_seconds() / 3600
earth_agreement = earth_agreement.drop(columns=["wind_start", "wind_shock"]).reset_index(drop=True)
show_table(earth_agreement)

,conjunction_id,conjunction_start,conjunction_end,matching_start,matching_end,sinp_cme_id,cme_type,sinp_field,sinp_time,wind_cme_id,wind_field,wind_time,sinp_minus_wind_hours
0,CONJ-012,2012-03-16 00:00:00,2012-03-29 00:00:00,2012-03-16 00:00:00,2012-03-29 00:00:00,SINP-902,CME,T_start,2012-03-15 17:00:00,ICME_Wind_NASA_20120315_01,mo_start_time,2012-03-15 16:04:00,0.933333
1,CONJ-018,2012-11-14 00:00:00,2012-11-21 00:00:00,2012-11-14 00:00:00,2012-11-21 00:00:00,SINP-938,CME,T_start,2012-11-13 06:00:00,ICME_Wind_NASA_20121112_01,mo_start_time,2012-11-13 08:23:00,-2.383333
2,CONJ-018,2012-11-14 00:00:00,2012-11-21 00:00:00,2012-11-14 00:00:00,2012-11-21 00:00:00,SINP-939,CME,T_start,2012-11-13 08:00:00,ICME_Wind_NASA_20121112_01,mo_start_time,2012-11-13 08:23:00,-0.383333
3,CONJ-024,2013-06-30 00:00:00,2013-07-18 00:00:00,2013-06-30 00:00:00,2013-07-18 00:00:00,SINP-975,CME,T_start,2013-07-05 01:00:00,ICME_Wind_NASA_20130704_01,mo_start_time,2013-07-05 04:05:00,-3.083333
4,CONJ-024,2013-06-30 00:00:00,2013-07-18 00:00:00,2013-06-30 00:00:00,2013-07-18 00:00:00,SINP-976,CME,T_start,2013-07-05 03:00:00,ICME_Wind_NASA_20130704_01,mo_start_time,2013-07-05 04:05:00,-1.083333
5,CONJ-024,2013-06-30 00:00:00,2013-07-18 00:00:00,2013-06-30 00:00:00,2013-07-18 00:00:00,SINP-978,Shock-Only CME,T_shock,2013-07-13 01:00:00,ICME_Wind_NASA_20130712_01,icme_start_time,2013-07-12 16:43:00,8.283333
6,CONJ-024,2013-06-30 00:00:00,2013-07-18 00:00:00,2013-06-30 00:00:00,2013-07-18 00:00:00,SINP-979,CME,T_start,2013-07-13 05:00:00,ICME_Wind_NASA_20130712_01,mo_start_time,2013-07-12 20:24:00,8.6
7,CONJ-031,2014-02-12 00:00:00,2014-03-06 00:00:00,2014-02-12 00:00:00,2014-02-21 00:00:00,SINP-1006,CME,T_start,2014-02-16 05:00:00,ICME_Wind_NASA_20140215_01,mo_start_time,2014-02-16 00:43:00,4.283333
8,CONJ-031,2014-02-12 00:00:00,2014-03-06 00:00:00,2014-02-12 00:00:00,2014-02-21 00:00:00,SINP-1008,CME,T_start,2014-02-18 15:00:00,ICME_Wind_NASA_20140219_01,mo_start_time,2014-02-19 03:09:00,-12.15
9,CONJ-031,2014-02-12 00:00:00,2014-03-06 00:00:00,2014-02-12 00:00:00,2014-02-21 00:00:00,SINP-1008,CME,T_start,2014-02-18 15:00:00,ICME_Wind_NASA_20140218_01,mo_start_time,2014-02-18 15:07:00,-0.116667


# Conjunction CME Table

One row represents one CME–conjunction match. Its MESSENGER or partner CME
columns are filled independently; the opposite side stays empty. Conjunctions
with no matched CMEs receive one empty-CME row. `partner_spacecraft` identifies
the partner observer on partner rows and lists the conjunction's participants
on MESSENGER or empty rows. VEX is displayed as Venus Express.

| Quantity | Source and units |
| --- | --- |
| CME start / end | HELIO4CAST `mo_start_time` / `mo_end_time`, or SINP `T_start` / `T_end`; UTC. |
| Bmax | HELIO4CAST `mo_bmax`, with missing Earth values supplemented by maximum hourly OMNI vector magnitude; nT. |
| Partner Vmean | HELIO4CAST `mo_speed_mean` or SINP `v_average_omni_body`; km/s. |
| Dst | SINP `dst_min_omni_body`, with missing Earth-CME values supplemented by minimum hourly OMNI Dst; nT. |

SINP's OMNI window has its producer's padding; shock-only CMEs use a 20-hour
post-shock window. Catalog speed means are used as provided. Missing Bmax/Dst supplements use
the hourly OMNI windows documented above; populated catalog values are retained. Mean speed
is unavailable for MESSENGER/VEX because the catalog has no plasma parameters there.

Earth Dst is `Missing` when unavailable for that row, and empty for non-Earth
partners. Earth Wind and SINP CMEs retain separate rows;
the existing catalog-agreement table provides possible correspondences.
Shock-only CMEs keep unknown body boundaries empty; their shock times and
matching fields remain in the working source table. **OMNI Supplements**
records the catalog IDs and table-row numbers for each supplemented Earth CME.

All table times are UTC, displayed and exported without timezone markers.

In [9]:
# Retain conjunctions with no CMEs as blank rows in the requested table.
unmatched = conjunctions[~conjunctions["conjunction_id"].isin(matches["conjunction_id"])]
table_sources = pd.concat([matches, unmatched.assign(role="No CME")], ignore_index=True)
table_sources = table_sources.sort_values(
    ["conjunction_start", "role", "observer", "cme_time"], na_position="last"
).reset_index(drop=True)

cme_table = table_sources[["conjunction_start", "conjunction_end"]].copy()
partner_rows = table_sources["role"].eq("Partner")
messenger_rows = table_sources["role"].eq("MESSENGER")
cme_table["partner_spacecraft"] = table_sources["observer"].replace({"VEX": "Venus Express"}).where(
    partner_rows, table_sources["partners"].str.replace("Venus", "Venus Express", regex=False)
)

# Separate rows avoid implying a MESSENGER–partner CME association.
for label, selection in (("messenger", messenger_rows), ("partner", partner_rows)):
    cme_table[f"{label}_cme_start"] = table_sources["cme_start"].where(selection)
    cme_table[f"{label}_cme_end"] = table_sources["cme_end"].where(selection)
    cme_table[f"{label}_cme_bmax"] = table_sources["bmax"].where(selection)
cme_table["partner_cme_vmean"] = table_sources["vmean"].where(partner_rows)
cme_table["dst"] = ""
earth_rows = partner_rows & table_sources["location"].eq("Earth")
earth_without_cme = ~partner_rows & table_sources["partners"].str.contains("Earth", regex=False)
cme_table.loc[earth_rows, "dst"] = table_sources.loc[earth_rows, "dst"].astype(object).fillna("Missing")
cme_table.loc[earth_without_cme, "dst"] = "Missing"

# CSV row order links the compact table to its provenance without extra columns.
table_sources.insert(0, "table_row", range(1, len(table_sources) + 1))
table_sources = table_sources[[
    "table_row", "conjunction_id", "role", "observer", "catalog", "cme_id", "cme_type",
    "match_field", "cme_time", "shock_time", "matching_start", "matching_end", "offset_hours",
    "bmax_source", "dst_source",
]]
show_table(cme_table)
omni_supplements = omni_supplements.merge(
    table_sources[["table_row", "conjunction_id", "catalog", "cme_id"]],
    on=["conjunction_id", "catalog", "cme_id"],
)

/var/folders/3z/b4fclst96kbd90mb019v618m0000gn/T/ipykernel_845/2439834335.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cme_table.loc[earth_rows, "dst"] = table_sources.loc[earth_rows, "dst"].astype(object).fillna("Missing")


,conjunction_start,conjunction_end,partner_spacecraft,messenger_cme_start,messenger_cme_end,messenger_cme_bmax,partner_cme_start,partner_cme_end,partner_cme_bmax,partner_cme_vmean,dst
0,2011-03-18 00:00:00,2011-03-20 00:00:00,STEREO-B,,,,,,,,
1,2011-04-03 00:00:00,2011-04-19 00:00:00,Earth (SINP),,,,2011-04-09 18:00:00,2011-04-10 02:00:00,4.633573,398.363636,-18.0
2,2011-05-17 00:00:00,2011-06-08 00:00:00,STEREO-A; Venus Express,2011-05-19 15:00:00,2011-05-20 10:36:00,72.6,,,,,
3,2011-05-17 00:00:00,2011-06-08 00:00:00,STEREO-A; Venus Express,2011-06-04 16:15:00,2011-06-05 00:43:00,118.4,,,,,
4,2011-05-17 00:00:00,2011-06-08 00:00:00,STEREO-A; Venus Express,2011-06-05 04:29:00,2011-06-05 12:45:00,169.9,,,,,
...,...,...,...,...,...,...,...,...,...,...,...
108,2014-10-12 00:00:00,2014-10-21 00:00:00,Earth,,,,,,,,Missing
109,2014-11-23 00:00:00,2015-01-05 00:00:00,STEREO-A; STEREO-B; Venus Express,2014-12-14 09:52:00,2014-12-14 21:41:00,90.7,,,,,
110,2014-11-23 00:00:00,2015-01-05 00:00:00,STEREO-A; STEREO-B; Venus Express,2014-12-14 23:37:00,2014-12-15 04:37:00,39.5,,,,,
111,2015-01-27 00:00:00,2015-02-04 00:00:00,Earth,,,,,,,,Missing


# Save Results

Update only the four existing files in `Conjunction Analysis`:

- **Conjunction CME Table.csv** — the requested table displayed above.
- **Catalog Summary.csv** — compact counts by observer and catalog.
- **OMNI Supplements.csv** — parameter windows, sample counts, fills, and sources.
- **Analysis Notes.md** — definitions, units, source details, and missingness rules.

All times are UTC and use `YYYY-MM-DD HH:MM:SS`, without timezone markers.
Working conjunction, participation, and source tables remain in memory.

In [10]:
# Keep the count summary for its existing CSV, while showing the CME table above.
catalog_summary = matches.groupby(["role", "observer", "catalog", "cme_type"], as_index=False).agg(
    cmes=("cme_id", "nunique"), conjunctions=("conjunction_id", "nunique"), matches=("cme_id", "size")
)
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)
tables = {
    "Conjunction CME Table.csv": cme_table,
    "Catalog Summary.csv": catalog_summary,
    "OMNI Supplements.csv": omni_supplements,
}
for filename, table in tables.items():
    table.to_csv(RESULTS_FOLDER / filename, index=False, date_format="%Y-%m-%d %H:%M:%S")

notes = f"""# Conjunction–CME Catalog Matching

All times are UTC, without timezone suffixes. Inputs are in ../Inputs.
A conjunction is one continuous run of input dates, even when partners change.
MESSENGER uses the whole conjunction; each partner uses its actual participation
windows. Matching tolerance is ±{TOLERANCE.total_seconds() / 3600:g} hours.

Conjunction CME Table.csv has one row per independent CME–conjunction match,
plus a blank-CME row for each unmatched conjunction. MESSENGER and partner
CMEs are not paired. Catalog Summary.csv holds unique CME and conjunction counts.

CME body boundaries are HELIO4CAST mo_start_time/mo_end_time or SINP T_start/T_end.
Shock-only SINP CMEs match on T_shock; their unknown body boundaries stay empty.
Bmax uses HELIO4CAST mo_bmax (nT). Missing Earth Bmax uses the maximum magnitude
of complete hourly OMNI GSE vectors, not a higher-cadence scalar magnetic peak.
Vmean is HELIO4CAST mo_speed_mean or SINP v_average_omni_body (km/s).
Dst is SINP dst_min_omni_body or, when missing for an Earth CME, minimum OMNI Dst (nT).

OMNI windows reproduce ICME_list's parameter windows: body start minus one hour
through body end plus one hour, inclusive. Shock-only windows run from shock
minus one hour through shock plus 21 hours. A known body start without an end
uses start plus 20 hours. OMNI Supplements.csv records windows, valid sample
counts, old/current values, fills, catalog IDs, and CME-table row numbers.
Existing populated values are preserved; the newer OMNI/full SINP snapshots
contain some revised Dst values relative to the older short SINP snapshot.
Rows without an Earth CME retain Missing Dst because no CME window is defined.

Inputs include messenger_conjunctions.csv, mercury_conj.xlsx, the original short
merged_icme_short.csv selection, the full merged_icme_omni.csv snapshot, the
complete HELIO4CAST catalog and per-spacecraft subsets, OMNI.parquet, and its
original omni_v_dst.pkl. Input Sources.csv and OMNI Input Sources.csv record provenance.
The OMNI parquet preserves the original samples without resampling or interpolation.

Only these four analysis files are written: Conjunction CME Table.csv,
Catalog Summary.csv, OMNI Supplements.csv, and Analysis Notes.md.
"""
(RESULTS_FOLDER / "Analysis Notes.md").write_text(notes)


2196